In [ ]:
import pandas as pd

# =========================================================
# 1. read file
# =========================================================
price = pd.read_excel(
    r"C:\Users\Dell\Desktop\Project\Price.xlsx"
)

weather = pd.read_excel(
    r"C:\Users\Dell\Desktop\Project\Weather.xlsx"
)

# =========================================================
# 2. convert date format
# =========================================================
price["Start date"] = pd.to_datetime(
    price["Start date"],
    errors="coerce", format="%Y-%m-%d %H:%M:%S"
)

price["End date"] = pd.to_datetime(
    price["End date"],
    errors="coerce", format="%Y-%m-%d %H:%M:%S"
)

weather["date"] = pd.to_datetime(
    weather["date"],
    errors="coerce"
)

print("Price rows:", len(price))
print("Weather rows:", len(weather))

# =========================================================
# 3. select date range
# =========================================================
weather = weather[
    (weather["date"] >= "2025-01-01") &
    (weather["date"] < "2026-09-30")
].copy()

price = price[
    (price["Start date"] >= "2025-01-01") &
    (price["Start date"] < "2026-09-30")
].copy()

print("filtered Weather rows:", len(weather))
print("Weather earliest date:", weather["date"].min())
print("Weather latest date:", weather["date"].max())

print("filtered Price rows:", len(price))
print("Price earliest date:", price["Start date"].min())
print("Price latest date:", price["Start date"].max())

# =========================================================
# 4. eliminate unrelated price list
# =========================================================
price = price[
    [
        "Start date",
        "Germany/Luxembourg [€/MWh]",
        "∅ DE/LU neighbours [€/MWh]"
    ]
].copy()

print(price.columns.tolist())
print("Price rows:", len(price))
print("Weather rows:", len(weather))

# =========================================================
# 5. check 2025-10-26 unusual timezone convertion
# =========================================================
print("\n--- Weather around 2025-10-26 ---")
print(
    weather[
        (weather["date"] >= "2025-10-26 00:00") &
        (weather["date"] <= "2025-10-26 04:00")
    ]["date"].tolist()
)

print("\n--- Price around 2025-10-26 ---")
print(
    price[
        (price["Start date"] >= "2025-10-26 00:00") &
        (price["Start date"] <= "2025-10-26 04:00")
    ]["Start date"].tolist()
)

# =========================================================
# 6. assign time order for matching（ 1-to-1 ）
# =========================================================
price_merge = price.copy()
weather_merge = weather.copy()

price_merge["time_order"] = (
    price_merge.groupby("Start date").cumcount()
)

weather_merge["time_order"] = (
    weather_merge.groupby("date").cumcount()
)

# =========================================================
# 7. merge
# =========================================================
merged = pd.merge(
    price_merge,
    weather_merge,
    left_on=["Start date", "time_order"],
    right_on=["date", "time_order"],
    how="inner"
)

merged = merged.reset_index(drop=True)

# delete duplicate dates column
merged = merged.drop(columns=["date", "time_order"])

print("\nMerged rows:", len(merged))
print(merged.head())

# =========================================================
# 8. export file after merge
# =========================================================
merged.to_excel(
    r"C:\Users\Dell\Desktop\Project\Merged.xlsx",
    index=False
)

print("\nSaved to Merged.xlsx")

# Merge Price and Weather Data

This notebook produces **two** merged output files:

| File | Contents | Used by |
|---|---|---|
| `Merged.xlsx` | DE/LU price + ∅ neighbour average + weather | Notebooks 4, 5, 6 |
| `Merged_with_neighbors.xlsx` | DE/LU price + **all neighbouring countries' prices** + weather | Notebooks 7, 8 |

The full version includes electricity prices from Belgium, Denmark 1 & 2, France, Netherlands, Norway 2, Austria, Poland, Sweden 4, Switzerland, Czech Republic, Northern Italy, Slovenia, and Hungary — for further market-coupling and forecasting analysis.

In [6]:
# =========================================================
# NOTE: This notebook produces TWO merged files:
#   (1) Merged.xlsx                 - DE/LU + ∅ neighbours only
#   (2) Merged_with_neighbors.xlsx  - includes ALL neighbouring
#                                     countries' electricity prices
#                                     for further analysis (notebooks 7 & 8)
# =========================================================

import pandas as pd

# =========================================================
# 1. Read files
# =========================================================
price = pd.read_excel(r"C:\Users\Dell\Desktop\Project\Price.xlsx")
weather = pd.read_excel(r"C:\Users\Dell\Desktop\Project\Weather.xlsx")

# =========================================================
# 2. Convert date format
# =========================================================
price["Start date"] = pd.to_datetime(price["Start date"], errors="coerce", format="%b %d, %Y %I:%M %p")
price["End date"]   = pd.to_datetime(price["End date"],   errors="coerce", format="%b %d, %Y %I:%M %p")
weather["date"]     = pd.to_datetime(weather["date"], errors="coerce")

print("Price rows:", len(price))
print("Weather rows:", len(weather))

# =========================================================
# 3. Select date range
# =========================================================
weather = weather[(weather["date"] >= "2025-01-01") & (weather["date"] < "2026-09-30")].copy()
price   = price[(price["Start date"] >= "2025-01-01") & (price["Start date"] < "2026-09-30")].copy()

print("Filtered Weather rows:", len(weather))
print("Filtered Price rows:", len(price))

# =========================================================
# 4a. Slim version (for notebooks 4 / 5 / 6)
# =========================================================
price_slim = price[
    [
        "Start date",
        "Germany/Luxembourg [€/MWh]",
        "∅ DE/LU neighbours [€/MWh]"
    ]
].copy()

# =========================================================
# 4b. Full version (for notebooks 7 / 8)
# =========================================================
price_full = price[
    [
        "Start date",
        "Germany/Luxembourg [€/MWh]",
        "∅ DE/LU neighbours [€/MWh]",
        "Belgium [€/MWh]",
        "Denmark 1 [€/MWh]",
        "Denmark 2 [€/MWh]",
        "France [€/MWh]",
        "Netherlands [€/MWh]",
        "Norway 2 [€/MWh]",
        "Austria [€/MWh]",
        "Poland [€/MWh]",
        "Sweden 4 [€/MWh]",
        "Switzerland [€/MWh]",
        "Czech Republic [€/MWh]",
        "Northern Italy [€/MWh]",
        "Slovenia [€/MWh]",
        "Hungary [€/MWh]"
    ]
].copy()

print("\nSlim columns:", price_slim.shape[1])
print("Full columns:", price_full.shape[1])

# =========================================================
# 5. Merge helper (1-to-1 by time_order)
# =========================================================
def merge_with_weather(price_df, weather_df):
    p = price_df.copy()
    w = weather_df.copy()
    p["time_order"] = p.groupby("Start date").cumcount()
    w["time_order"] = w.groupby("date").cumcount()
    merged = pd.merge(
        p, w,
        left_on=["Start date", "time_order"],
        right_on=["date", "time_order"],
        how="inner"
    )
    return merged.drop(columns=["date", "time_order"]).reset_index(drop=True)

# =========================================================
# 6. Merge both versions
# =========================================================
merged_slim = merge_with_weather(price_slim, weather)
merged_full = merge_with_weather(price_full, weather)

# ---------- Drop DST duplicates ----------
# 2025-10-26 02:00–02:45 appears twice due to daylight saving switch.
# Keep first occurrence to match previous dataset (61144 rows).
merged_slim = merged_slim.drop_duplicates(subset="Start date", keep="first").reset_index(drop=True)
merged_full = merged_full.drop_duplicates(subset="Start date", keep="first").reset_index(drop=True)

print("\nMerged (slim) rows:", len(merged_slim), "| columns:", merged_slim.shape[1])
print("Merged (full) rows:", len(merged_full), "| columns:", merged_full.shape[1])

# =========================================================
# 7. Export
# =========================================================
merged_slim.to_excel(r"C:\Users\Dell\Desktop\Project\Merged.xlsx", index=False)
merged_full.to_excel(r"C:\Users\Dell\Desktop\Project\Merged_with_neighbors.xlsx", index=False)

print("\nSaved: Merged.xlsx (slim)")
print("Saved: Merged_with_neighbors.xlsx (full)")

Price rows: 61220
Weather rows: 61248
Filtered Weather rows: 61148
Filtered Price rows: 61148

Slim columns: 3
Full columns: 17

Merged (slim) rows: 61144 | columns: 10
Merged (full) rows: 61144 | columns: 24

Saved: Merged.xlsx (slim)
Saved: Merged_with_neighbors.xlsx (full)


In [ ]:
m = pd.read_excel(r"C:\Users\Dell\Desktop\Project\Merged.xlsx")
m["Start date"] = pd.to_datetime(m["Start date"])

# Check DST
dst = m[(m["Start date"] >= "2025-10-26 01:30") & (m["Start date"] <= "2025-10-26 03:00")]
print(dst[["Start date"]].to_string())